[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C76_Causal_Inference_Course/02_dag_backdoor/02_dag_backdoor.ipynb)

# C76 模块 02 · 因果图与识别

四件事：

1. **对撞偏差**：把 $X$ 对 $Y$ 的系数从 $+0.0006$ 变成 $-0.917$；
2. **控制变量枚举表**：$\{X\}$ 无偏，$\{X,C\}$ 偏差 $-2.596$；
3. **M-bias**：控制一个*处理前*变量把偏差从 $+0.001$ 变成 $-0.312$；
4. 把 **Bayes-Ball 与后门准则实现成代码**，枚举 8 个子集，只有 1 个合法。

纯 numpy / CPU / 离线。

In [ ]:
import numpy as np
import itertools

def ols(y, ctrl):
    '''回归 y ~ 1 + ctrl[0] + ctrl[1] + ...，返回系数向量（含截距）。'''
    A = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in ctrl])
    return np.linalg.lstsq(A, y, rcond=None)[0]

N = 600_000
print(f'样本量 N = {N:,}（够大，让偏差与噪声可区分）')

## 1. 三种基本结构

```
① 链    A -> B -> C      控制 B 阻断
② 叉    A <- B -> C      控制 B 阻断
③ 对撞  A -> B <- C      控制 B **打开**
```

前两种符合「控制掉就干净了」的直觉，第三种反过来。

In [ ]:
r = np.random.default_rng(0)

# ① 链 A -> B -> C
A1 = r.normal(0, 1, N); B1 = A1 + r.normal(0, 1, N); C1 = B1 + r.normal(0, 1, N)
# ② 叉 A <- B -> C
B2 = r.normal(0, 1, N); A2 = B2 + r.normal(0, 1, N); C2 = B2 + r.normal(0, 1, N)
# ③ 对撞 A -> B <- C
A3 = r.normal(0, 1, N); C3 = r.normal(0, 1, N); B3 = A3 + C3 + r.normal(0, 0.3, N)

def rel(a, c, b=None):
    '''a 与 c 的关系强度：不控制 b 时用相关，控制 b 时用偏回归系数。'''
    if b is None:
        return float(np.corrcoef(a, c)[0, 1])
    return float(ols(c, [a, b])[1])

print('结构        不控制 B          控制 B')
print(f'① 链       {rel(A1,C1):+.5f}          {rel(A1,C1,B1):+.5f}   <- 阻断')
print(f'② 叉       {rel(A2,C2):+.5f}          {rel(A2,C2,B2):+.5f}   <- 阻断')
print(f'③ 对撞     {rel(A3,C3):+.5f}          {rel(A3,C3,B3):+.5f}   <- 打开！')

assert abs(rel(A1, C1)) > 0.4 and abs(rel(A1, C1, B1)) < 0.01
assert abs(rel(A2, C2)) > 0.4 and abs(rel(A2, C2, B2)) < 0.01
assert abs(rel(A3, C3)) < 0.01 and abs(rel(A3, C3, B3)) > 0.8
print()
print('-> 前两种「控制掉就干净了」，第三种「控制掉才脏」。')

## 2. 对撞偏差的定量：条件相关有闭式解

在联合正态下

$$\text{Cov}(X, Y \mid Z) = \text{Cov}(X,Y) - \frac{\text{Cov}(X,Z)\,\text{Cov}(Z,Y)}{\text{Var}(Z)}$$

取 $Z = X + Y + \varepsilon$，$\varepsilon \sim N(0,\sigma^2)$，则条件相关 $= -1/(1+\sigma^2)$。

In [ ]:
r = np.random.default_rng(1)
X = r.normal(0, 1, N)
Y = r.normal(0, 1, N)
sigma = 0.3
Z = X + Y + r.normal(0, sigma, N)

print(f'X 与 Y 在构造上完全独立（无任何因果关系）')
print(f'  边际 corr(X, Y)        = {np.corrcoef(X, Y)[0,1]:+.5f}')
print(f'  回归 Y ~ X 的系数      = {ols(Y, [X])[1]:+.5f}')
print(f'  回归 Y ~ X + Z 的系数  = {ols(Y, [X, Z])[1]:+.5f}   <- 凭空出现')
print()
band = np.abs(Z) < 0.2
print(f'  限定在 |Z| < 0.2 的 {band.sum():,} 个样本内:')
print(f'    corr(X, Y | 该带内)  = {np.corrcoef(X[band], Y[band])[0,1]:+.5f}')
print('    -> 分层看同样出现，所以这不是回归模型的锅，是条件化本身的效果。')
print()
theory = -1.0 / (1.0 + sigma**2)
emp = float(np.corrcoef(X[band], Y[band])[0, 1])
print(f'  闭式解 -1/(1+sigma^2) = {theory:+.5f}')
print(f'  实测（|Z|<0.2 带内）  = {emp:+.5f}')
print(f'  差 {abs(theory-emp):.4f}')
assert abs(ols(Y, [X])[1]) < 0.01, '边际上应无关'
assert ols(Y, [X, Z])[1] < -0.8, '控制对撞后应出现强负关联'
assert abs(theory - emp) < 0.02, '闭式解应与实测一致'
print()
print('-> 强度可以精确预测：sigma 越小（Z 越接近 X+Y），负关联越强。')
for s in (0.1, 0.3, 1.0, 3.0):
    print(f'   sigma={s:4.1f} -> 理论条件相关 {-1/(1+s**2):+.4f}')

## 3. 把 Bayes-Ball 与后门准则实现成代码

d-分离的判定用 Shachter (1998) 的 Bayes-Ball 算法：
从 $X$ 出发做 BFS，状态是「节点 + 来向」，按三种结构决定能否继续传播。

In [ ]:
class DAG:
    '''最小 DAG：d-分离（Bayes-Ball）与后门准则。'''

    def __init__(self, edges):
        self.nodes = sorted({v for e in edges for v in e})
        self.par = {v: set() for v in self.nodes}
        self.ch  = {v: set() for v in self.nodes}
        for a, b in edges:
            self.par[b].add(a)
            self.ch[a].add(b)

    def ancestors(self, S):
        out, stack = set(S), list(S)
        while stack:
            v = stack.pop()
            for p in self.par.get(v, ()):
                if p not in out:
                    out.add(p); stack.append(p)
        return out

    def descendants(self, v):
        out, stack = set(), [v]
        while stack:
            u = stack.pop()
            for c in self.ch.get(u, ()):
                if c not in out:
                    out.add(c); stack.append(c)
        return out

    def d_sep(self, Xs, Ys, Zs):
        '''Z 是否 d-分离 X 与 Y。'''
        Xs, Ys, Zs = set(Xs), set(Ys), set(Zs)
        anc_Z = self.ancestors(Zs)
        seen, frontier = set(), [(x, 'up') for x in Xs]
        while frontier:
            v, d = frontier.pop()
            if (v, d) in seen:
                continue
            seen.add((v, d))
            if v in Ys:
                return False                  # 找到一条活跃路径
            if d == 'up':                     # 从子节点传上来
                if v not in Zs:
                    frontier += [(p, 'up') for p in self.par.get(v, ())]
                    frontier += [(c, 'down') for c in self.ch.get(v, ())]
            else:                             # 从父节点传下来
                if v not in Zs:
                    frontier += [(c, 'down') for c in self.ch.get(v, ())]
                if v in anc_Z:                # 对撞（或其祖先在 Z 里）-> 打开
                    frontier += [(p, 'up') for p in self.par.get(v, ())]
        return True

    def backdoor_ok(self, T, Y, Zs):
        '''后门准则：① Z 不含 T 的后代；② 删掉 T 的出边后 T ⊥ Y | Z。'''
        if set(Zs) & self.descendants(T):
            return False
        edges2 = [(a, b) for a in self.nodes for b in self.ch[a] if a != T]
        g2 = DAG(edges2) if edges2 else None
        if g2 is None:
            return True
        for v in self.nodes:                  # 补上被删成孤立的节点
            if v not in g2.par:
                g2.nodes.append(v); g2.par[v] = set(); g2.ch[v] = set()
        return g2.d_sep({T}, {Y}, set(Zs))

# 先在三种基本结构上验证 d_sep
g_chain = DAG([('A','B'), ('B','C')])
g_fork  = DAG([('B','A'), ('B','C')])
g_coll  = DAG([('A','B'), ('C','B')])
assert not g_chain.d_sep({'A'}, {'C'}, set())   and g_chain.d_sep({'A'}, {'C'}, {'B'})
assert not g_fork.d_sep({'A'}, {'C'}, set())    and g_fork.d_sep({'A'}, {'C'}, {'B'})
assert g_coll.d_sep({'A'}, {'C'}, set())    and not g_coll.d_sep({'A'}, {'C'}, {'B'})
print('✅ Bayes-Ball 在三种基本结构上与第 1 节的数值一致')

# 对撞的后代也会部分打开
g_cd = DAG([('A','B'), ('C','B'), ('B','D')])
assert g_cd.d_sep({'A'}, {'C'}, set())
assert not g_cd.d_sep({'A'}, {'C'}, {'D'})
print('✅ 控制对撞的**后代** D 同样打开路径（这是准则第 ① 条的由来）')

## 4. 枚举全部子集：只有一个合法

```
        X -> T, X -> Y      X 是混杂
        T -> Y              直接效应 2.0
        T -> M, M -> Y      M 是中介，经它的效应 0.9
        T -> C, Y -> C      C 是对撞
        总效应 = 2.9
```

In [ ]:
EDGES = [('X','T'), ('X','Y'), ('T','Y'), ('T','M'), ('M','Y'), ('T','C'), ('Y','C')]
g = DAG(EDGES)
OBS = ['X', 'M', 'C']

def gen_dag_data(n=N, seed=1):
    r = np.random.default_rng(seed)
    Xv = r.normal(0, 1, n)
    Tv = 0.8 * Xv + r.normal(0, 1, n)
    Mv = 0.9 * Tv + r.normal(0, 1, n)
    Yv = 2.0 * Tv + 1.0 * Mv + 1.5 * Xv + r.normal(0, 1, n)
    Cv = 1.0 * Tv + 1.0 * Yv + r.normal(0, 1, n)
    return dict(X=Xv, T=Tv, M=Mv, Y=Yv, C=Cv)

D = gen_dag_data()
TOTAL, DIRECT = 2.9, 2.0
print(f'真总效应 = {TOTAL}（直接 {DIRECT} + 经 M 的 0.9*1.0 = 0.9）')
print()
print('  子集           后门准则     回归系数      与总效应之差')
results = {}
for k in range(len(OBS) + 1):
    for sub in itertools.combinations(OBS, k):
        ok = g.backdoor_ok('T', 'Y', set(sub))
        coef = float(ols(D['Y'], [D['T']] + [D[v] for v in sub])[1])
        results[sub] = (ok, coef)
        name = '{' + ', '.join(sub) + '}' if sub else '{}'
        print(f'  {name:14s} {"✓ 合法" if ok else "✗ 不合法":10s}  '
              f'{coef:+9.5f}    {coef-TOTAL:+8.5f}')

legal = [s for s, (ok, _) in results.items() if ok]
print()
print(f'✅ {len(results)} 个子集中，后门准则判定合法的只有 {len(legal)} 个：{legal}')
assert legal == [('X',)], f'应只有 {{X}} 合法，得到 {legal}'
assert abs(results[('X',)][1] - TOTAL) < 0.01, '{X} 应无偏估计总效应'

# 准则与数值一致性：合法 <=> 无偏
for sub, (ok, coef) in results.items():
    unbiased = abs(coef - TOTAL) < 0.05
    assert ok == unbiased, f'{sub}: 准则说 {ok}，数值说无偏={unbiased}'
print('✅ 「后门准则合法」与「回归无偏」在全部 8 个子集上完全一致')

In [ ]:
# {X, M} 那一行不是「不太准」，它精确地估出了**直接效应**
c_xm = results[('X', 'M')][1]
print(f'{{X, M}} 的系数 = {c_xm:+.5f}')
print(f'真直接效应      = {DIRECT}')
print(f'差              = {abs(c_xm - DIRECT):.5f}')
assert abs(c_xm - DIRECT) < 0.01, '控制中介应无偏估计直接效应'
print()
print('-> 它答对了**另一个问题**。经由 M 的 0.9 凭空消失了，')
print(f'   占总效应的 {0.9/TOTAL*100:.0f}%，而没有任何统计信号提示出错：')
print('   系数稳定、置信区间正常、换 seed 也复现。')
print()
print('换 seed 复现:')
for s in (2, 3, 4):
    Ds = gen_dag_data(seed=s)
    print(f'  seed={s}: {{X}} -> {ols(Ds["Y"], [Ds["T"], Ds["X"]])[1]:+.5f}   '
          f'{{X,M}} -> {ols(Ds["Y"], [Ds["T"], Ds["X"], Ds["M"]])[1]:+.5f}')

## 5. M-bias：控制一个「处理前」变量制造偏差

```
    U1 -> T,  U1 -> Z,  U2 -> Z,  U2 -> Y,  T -> Y (真效应 1.0)

    U1, U2 未观测；Z 在处理**之前**测到
    U1 不影响 Y、U2 不影响 T -> T 与 Y 之间本来没有混杂
```

In [ ]:
r = np.random.default_rng(3)
U1 = r.normal(0, 1, N)
U2 = r.normal(0, 1, N)
Zm = 1.0 * U1 + 1.0 * U2 + r.normal(0, 0.3, N)   # 对撞，且在处理前
Tm = 1.0 * U1 + r.normal(0, 1, N)
Ym = 1.0 * Tm + 1.0 * U2 + r.normal(0, 1, N)

print(f'Z 与 T 的相关 = {np.corrcoef(Zm, Tm)[0,1]:+.4f}')
print('  -> Z 与 T 明显相关，看起来完全像一个该控制的基线协变量')
print()
print('  控制的变量集              T 的系数      偏差')
mb = {}
for tag, ctrl in [('{} 不控制', []), ('{Z} 处理前变量', [Zm]),
                  ('{Z, U1}', [Zm, U1]), ('{Z, U2}', [Zm, U2]),
                  ('{U1, U2}', [U1, U2])]:
    c = float(ols(Ym, [Tm] + ctrl)[1])
    mb[tag] = c
    print(f'  {tag:24s} {c:+9.5f}   {c-1.0:+.5f}')

assert abs(mb['{} 不控制'] - 1.0) < 0.01, '不控制时应无偏'
assert mb['{Z} 处理前变量'] < 0.75, 'M-bias 应使系数明显下偏'
assert abs(mb['{Z, U1}'] - 1.0) < 0.01, '补上 U1 应修好'
assert abs(mb['{Z, U2}'] - 1.0) < 0.01, '补上 U2 应修好'

print()
print(f'✅ 不控制时无偏（{mb["{} 不控制"]:.5f}），')
print(f'   而控制这个**处理前**变量后偏差是 {mb["{Z} 处理前变量"]-1.0:+.5f}'
      f'（真效应的 {abs(mb["{Z} 处理前变量"]-1.0)*100:.0f}%）')
print()
print('   后两行给出修补方式：补上 U1 或 U2 任意一个即可。')
print('   但它们按设定是**未观测**的 —— 你能看见制造问题的那个，')
print('   看不见能修好它的那两个。')
print()
print('   后门准则怎么判：')
g_m = DAG([('U1','T'), ('U1','Z'), ('U2','Z'), ('U2','Y'), ('T','Y')])
for zs in [set(), {'Z'}, {'Z','U1'}, {'Z','U2'}, {'U1','U2'}]:
    ok = g_m.backdoor_ok('T', 'Y', zs)
    print(f'     Z={str(sorted(zs)) if zs else "[]":16s} -> {"✓ 合法" if ok else "✗ 不合法"}')
assert g_m.backdoor_ok('T', 'Y', set()), '空集应合法（本来无混杂）'
assert not g_m.backdoor_ok('T', 'Y', {'Z'}), '{Z} 应不合法'

## ✏️ 练习 1：判断一个变量是好控制还是坏控制

给定 DAG、处理 $T$、结果 $Y$ 和单个变量 $v$，
实现 `classify_control(g, T, Y, v)`，返回下列之一：

- `'confounder'`：$\{v\}$ 满足后门准则（该控制）
- `'collider'`：$v$ 同时是 $T$ 和 $Y$ 的后代（绝不控制）
- `'mediator'`：$v$ 是 $T$ 的后代且是 $Y$ 的祖先（控制会改成直接效应）
- `'other'`：其余

In [ ]:
def classify_control(g, T, Y, v):
    '''把单个变量 v 分类为 confounder / collider / mediator / other。

    判定顺序很重要：先判 collider（它也可能同时是 mediator 的后代），
    再判 mediator，最后用后门准则判 confounder。

    参数
    ----
    g : DAG 实例
    T : 处理节点名
    Y : 结果节点名
    v : 待分类的节点名

    返回
    ----
    str : 'confounder' | 'collider' | 'mediator' | 'other'
    '''
    # TODO: 用 g.descendants / g.ancestors / g.backdoor_ok 实现上述四类判定
    raise NotImplementedError

In [ ]:
# 自测
_g = DAG(EDGES)          # X->T, X->Y, T->Y, T->M, M->Y, T->C, Y->C
_expect = {'X': 'confounder', 'M': 'mediator', 'C': 'collider'}
for _v, _e in _expect.items():
    _got = classify_control(_g, 'T', 'Y', _v)
    assert _got == _e, f'{_v}: 期望 {_e}，得到 {_got}'
    print(f'  {_v} -> {_got}')

# M-bias 图里 Z 既不是混杂也不是对撞（对 T,Y 而言），应归入 other
_gm = DAG([('U1','T'), ('U1','Z'), ('U2','Z'), ('U2','Y'), ('T','Y')])
_gz = classify_control(_gm, 'T', 'Y', 'Z')
assert _gz == 'other', f'M-bias 的 Z 应为 other，得到 {_gz}'
print(f'  M-bias 的 Z -> {_gz}   <- 注意它**不是**对撞（对 T,Y 而言），却依然不该控制')

# 纯结果预测变量：P -> Y，与 T 无关
_gp = DAG([('T','Y'), ('P','Y')])
assert classify_control(_gp, 'T', 'Y', 'P') == 'other'
print(f'  纯结果预测变量 P -> other')

print()
print('✅ 分类器与第 4、5 节的数值一致。')
print('   注意 other 类里同时包含「无害且降方差」（P）与「有害」（M-bias 的 Z）——')
print('   所以单变量分类**不足以**决定去留，必须看整个调整集。')

## ✏️ 练习 2：枚举全部合法调整集

实现 `all_valid_sets(g, T, Y, candidates)`，返回 `candidates` 的
全部满足后门准则的子集（按大小、再按字典序排序）。

In [ ]:
def all_valid_sets(g, T, Y, candidates):
    '''枚举 candidates 的全部合法调整集。

    参数
    ----
    g          : DAG 实例
    T, Y       : 处理与结果节点名
    candidates : 可观测的候选变量列表

    返回
    ----
    list[tuple] : 全部满足后门准则的子集，按 (大小, 字典序) 排序
    '''
    # TODO: 用 itertools.combinations 枚举全部子集，用 g.backdoor_ok 筛选
    raise NotImplementedError

In [ ]:
# 自测
_v1 = all_valid_sets(DAG(EDGES), 'T', 'Y', ['X', 'M', 'C'])
assert _v1 == [('X',)], f'该图应只有 {{X}} 合法，得到 {_v1}'
print(f'  图 1（X 混杂 / M 中介 / C 对撞）: {_v1}   ({len(_v1)}/8)')

# 两个混杂：X1, X2 各自不够，需要同时控制
_g2 = DAG([('X1','T'), ('X1','Y'), ('X2','T'), ('X2','Y'), ('T','Y')])
_v2 = all_valid_sets(_g2, 'T', 'Y', ['X1', 'X2'])
assert _v2 == [('X1', 'X2')], f'两个混杂需同时控制，得到 {_v2}'
print(f'  图 2（两个独立混杂）:              {_v2}   ({len(_v2)}/4)')

# 混杂经由一个可观测代理：控制 X 或 W 都行
_g3 = DAG([('X','T'), ('X','W'), ('W','Y'), ('T','Y')])
_v3 = all_valid_sets(_g3, 'T', 'Y', ['X', 'W'])
assert {frozenset(x) for x in _v3} == {frozenset(('X',)), frozenset(('W',)),
                                        frozenset(('X', 'W'))}, f'三个都该合法，得到 {_v3}'
print(f'  图 3（混杂经代理 W 影响 Y）:        {_v3}   ({len(_v3)}/4)')

# 无混杂：空集合法，加变量也可能合法
_g4 = DAG([('T','Y'), ('P','Y')])
_v4 = all_valid_sets(_g4, 'T', 'Y', ['P'])
assert () in _v4 and ('P',) in _v4
print(f'  图 4（无混杂 + 纯预测变量 P）:      {_v4}   ({len(_v4)}/2)')

print()
print('✅ 合法集的个数在四个图上分别是 1/8, 1/4, 3/4, 2/2 —— 差异极大。')
print('   图 3 说明合法集可以有多个：此时应选**方差最小**的那个（模块 03）。')
print('   图 1 说明也可以几乎没有：此时靠猜的成功率是 1/8 = 12.5%。')

## ✏️ 练习 3：量化控制中介造成的效应损失

实现 `mediated_share(direct, a_tm, b_my)`：给定直接效应、$T\to M$ 与
$M\to Y$ 的系数，返回**经由中介的效应占总效应的比例**。

控制中介会让这个比例的效应从报告里消失。

In [ ]:
def mediated_share(direct, a_tm, b_my):
    '''经由中介的效应占总效应的比例。

    总效应 = direct + a_tm * b_my

    参数
    ----
    direct : T -> Y 的直接效应
    a_tm   : T -> M 的系数
    b_my   : M -> Y 的系数

    返回
    ----
    float : (a_tm * b_my) / 总效应
    '''
    # TODO
    raise NotImplementedError

In [ ]:
# 自测：先用第 4 节的参数验证，再用模拟交叉验证
_share = mediated_share(2.0, 0.9, 1.0)
assert abs(_share - 0.9/2.9) < 1e-12, f'应为 0.9/2.9，得到 {_share}'
print(f'  第 4 节的参数: 经中介的份额 = {_share*100:.1f}%')

def _sim(direct, a_tm, b_my, n=400_000, seed=1):
    '''模拟并返回 (总效应估计, 直接效应估计)。'''
    r = np.random.default_rng(seed)
    Xv = r.normal(0, 1, n)
    Tv = 0.8 * Xv + r.normal(0, 1, n)
    Mv = a_tm * Tv + r.normal(0, 1, n)
    Yv = direct * Tv + b_my * Mv + 1.5 * Xv + r.normal(0, 1, n)
    tot = float(ols(Yv, [Tv, Xv])[1])
    dir_ = float(ols(Yv, [Tv, Xv, Mv])[1])
    return tot, dir_

print()
print('  direct  a_tm  b_my   预测份额   模拟 (总-直接)/总   差')
for _d, _a, _b in [(2.0, 0.9, 1.0), (1.0, 2.0, 1.0), (0.5, 1.0, 3.0), (3.0, 0.2, 0.5)]:
    _pred = mediated_share(_d, _a, _b)
    _tot, _dir = _sim(_d, _a, _b)
    _emp = (_tot - _dir) / _tot
    assert abs(_pred - _emp) < 0.01, f'预测 {_pred:.4f} vs 模拟 {_emp:.4f}'
    print(f'  {_d:5.1f}  {_a:4.1f}  {_b:4.1f}   {_pred*100:7.1f}%   {_emp*100:14.1f}%   {abs(_pred-_emp):.4f}')

assert mediated_share(1.0, 0.0, 5.0) == 0.0, '无 T->M 时份额为 0'
assert mediated_share(0.0, 1.0, 1.0) == 1.0, '无直接效应时份额为 1'
print()
print('✅ 预测与模拟一致。极端情形：a_tm=0 时份额 0，direct=0 时份额 100%')
print('   -> 若效应**全部**经由中介，控制中介会把报告的效应变成 0，')
print('      而这在数值上表现为「该功能没有效果」，不会表现为任何错误。')

## ✏️ 练习 4：M-bias 的强度取决于什么

M-bias 的大小由图上的四个系数决定。
实现 `mbias_magnitude(a1, a2, b1, b2, sz)`，用**模拟**返回控制 $Z$ 后的偏差：

```
U1 -> T (系数 a1),  U1 -> Z (b1)
U2 -> Z (b2),       U2 -> Y (a2)
T  -> Y (真效应 1.0),  Z 的噪声 sd = sz
```

用它找出偏差最大与最小的参数组合。

In [ ]:
def mbias_magnitude(a1, a2, b1, b2, sz=0.3, n=300_000, seed=3):
    '''控制 Z 后 T 系数的偏差（相对真效应 1.0）。

    参数
    ----
    a1 : U1 -> T
    a2 : U2 -> Y
    b1 : U1 -> Z
    b2 : U2 -> Z
    sz : Z 的噪声标准差
    n, seed : 模拟规模与随机种子

    返回
    ----
    float : ols(Y, [T, Z]) 的 T 系数 - 1.0
    '''
    # TODO: 按上面的 DAG 生成 U1, U2, Z, T, Y（T->Y 真效应 1.0），
    #       返回 ols(Y, [T, Z])[1] - 1.0
    raise NotImplementedError

In [ ]:
# 自测
# ① 基准：复现第 5 节的数值
_b0 = mbias_magnitude(1.0, 1.0, 1.0, 1.0, sz=0.3)
print(f'  基准 (a1=a2=b1=b2=1, sz=0.3): 偏差 = {_b0:+.5f}')
assert -0.40 < _b0 < -0.25, f'应约为 -0.31，得到 {_b0:.5f}'

# ② 断开任一条边，M-bias 消失
print()
print('  断开一条边:')
for _tag, _kw in [('a1=0 (U1 不影响 T)', dict(a1=0.0)),
                  ('a2=0 (U2 不影响 Y)', dict(a2=0.0)),
                  ('b1=0 (U1 不影响 Z)', dict(b1=0.0)),
                  ('b2=0 (U2 不影响 Z)', dict(b2=0.0))]:
    _kwargs = dict(a1=1.0, a2=1.0, b1=1.0, b2=1.0)
    _kwargs.update(_kw)
    _b = mbias_magnitude(**_kwargs)
    print(f'    {_tag:22s} 偏差 = {_b:+.5f}')
    assert abs(_b) < 0.02, f'{_tag}: 断开后 M-bias 应消失，得到 {_b:.5f}'

# ③ Z 的噪声越大，M-bias 越弱（Z 对两个 U 的信息越少）
print()
print('  Z 的噪声 sz 的影响:')
_prev = None
for _sz in (0.1, 0.3, 1.0, 3.0, 10.0):
    _b = mbias_magnitude(1.0, 1.0, 1.0, 1.0, sz=_sz)
    print(f'    sz={_sz:5.1f}  偏差 = {_b:+.5f}')
    if _prev is not None:
        assert _b > _prev - 1e-6, f'噪声越大偏差应越接近 0：sz={_sz}'
    _prev = _b
assert abs(mbias_magnitude(1.0, 1.0, 1.0, 1.0, sz=10.0)) < 0.05, \
    'Z 噪声极大时 Z 几乎不含 U 的信息，M-bias 应消失'

print()
print('✅ M-bias 需要**四条边同时存在**，断任一条即消失。')
print('   这解释了为什么它在实践中的量级常常很小：')
print('   它要求 Z 同时是两个混杂的强后代，而这四个系数的乘积很容易接近 0。')
print('   -> 本节的论点不是「不要控制处理前变量」，')
print('      而是「时间先后不是判据」—— 判据只能是图。')

## 📖 参考答案

In [ ]:
def classify_control(g, T, Y, v):
    '''把单个变量 v 分类为 confounder / collider / mediator / other。'''
    desc_T = g.descendants(T)
    desc_Y = g.descendants(Y)
    if v in desc_T and v in desc_Y:
        return 'collider'
    if v in desc_T and Y in g.descendants(v):
        return 'mediator'
    if g.backdoor_ok(T, Y, {v}) and not g.backdoor_ok(T, Y, set()):
        return 'confounder'
    return 'other'

def all_valid_sets(g, T, Y, candidates):
    '''枚举 candidates 的全部合法调整集。'''
    out = []
    for k in range(len(candidates) + 1):
        for sub in itertools.combinations(sorted(candidates), k):
            if g.backdoor_ok(T, Y, set(sub)):
                out.append(sub)
    return sorted(out, key=lambda s: (len(s), s))

def mediated_share(direct, a_tm, b_my):
    '''经由中介的效应占总效应的比例。'''
    med = a_tm * b_my
    total = direct + med
    if total == 0:
        return 0.0
    return float(med / total)

def mbias_magnitude(a1, a2, b1, b2, sz=0.3, n=300_000, seed=3):
    '''控制 Z 后 T 系数的偏差。'''
    r = np.random.default_rng(seed)
    u1 = r.normal(0, 1, n)
    u2 = r.normal(0, 1, n)
    z  = b1 * u1 + b2 * u2 + r.normal(0, sz, n)
    t  = a1 * u1 + r.normal(0, 1, n)
    y  = 1.0 * t + a2 * u2 + r.normal(0, 1, n)
    return float(ols(y, [t, z])[1] - 1.0)

print('参考答案已定义。')
print()
print('要点：')
print('  1. classify_control 里判定**顺序**是必要的：collider 先于 mediator，')
print('     因为一个节点可以同时是某条中介链的后代和某个对撞。')
print('  2. 「合法调整集」的个数在不同图上从 1/8 到 2/2 —— 靠直觉挑的成功率不可预期。')
print('  3. 控制中介损失的效应份额 = a_tm*b_my/总效应，可以精确预测。')
print('  4. M-bias 需要四条边同时存在；它的实践意义不是禁令，而是「时间不是判据」。')

## 🧪 真实工程胶囊：把 DAG 写进代码仓库

下面这段代码把「该控制哪些变量」从一次性的讨论变成一个**可执行的断言**：
DAG 用一个显式的边列表声明在代码里，特征工程从它推导，
而 CI 可以在图变化时提醒相关的估计代码需要复核。

关键设计：`adjustment_set()` **不接受**手工指定的变量列表——
它只接受图和可观测变量集，然后自己算。
这样「往模型里多加一个特征」就不再是一行改动，
而必须先在图里说明这个特征的位置。

In [ ]:
SPEC = {
    # 一个简化的推荐位实验的因果假设（真实项目里这段应该有注释说明每条边的依据）
    'edges': [
        ('user_tenure',   'exposure'),      # 老用户更可能进灰度（分桶按注册时间）
        ('user_tenure',   'engagement'),    # 老用户本身更活跃
        ('exposure',      'ctr'),           # 曝光 -> 点击率（中介）
        ('ctr',           'engagement'),    # 点击 -> 参与度
        ('exposure',      'engagement'),    # 直接效应
        ('engagement',    'support_ticket'),# 参与度 -> 客服工单
        ('exposure',      'support_ticket'),# 曝光 -> 客服工单（对撞的一半）
        ('device_tier',   'engagement'),    # 纯结果预测变量
    ],
    'treatment': 'exposure',
    'outcome':   'engagement',
    'observable': ['user_tenure', 'ctr', 'support_ticket', 'device_tier'],
}

def adjustment_set(spec, prefer='min_size'):
    '''从图推导调整集。不接受手工指定的变量列表。'''
    gg = DAG(spec['edges'])
    valid = all_valid_sets(gg, spec['treatment'], spec['outcome'], spec['observable'])
    if not valid:
        raise ValueError('该图下不存在合法调整集：必须换设计（模块 04）或补数据')
    chosen = valid[0] if prefer == 'min_size' else valid[-1]
    return gg, valid, chosen

gg, valid, chosen = adjustment_set(SPEC)
print(f'候选可观测变量: {SPEC["observable"]}')
print(f'合法调整集共 {len(valid)} 个（候选子集共 {2**len(SPEC["observable"])} 个）:')
for v in valid:
    print(f'    {v}')
print()
print(f'选用（最小）: {chosen}')
print()
print('每个候选变量的角色:')
for v in SPEC['observable']:
    role = classify_control(gg, SPEC['treatment'], SPEC['outcome'], v)
    verdict = {'confounder': '必须控制', 'mediator': '不要控制（会变成直接效应）',
               'collider': '绝不控制', 'other': '看整个调整集'}[role]
    print(f'  {v:16s} {role:11s} {verdict}')

print()
print('CI 断言（图变化时这些会失败，提示复核）:')
assert 'user_tenure' in chosen, 'user_tenure 是混杂，必须在调整集里'
assert 'ctr' not in chosen, 'ctr 是中介，控制它会把总效应换成直接效应'
assert 'support_ticket' not in chosen, 'support_ticket 是对撞，绝不能控制'
print('  ✅ 混杂在集内；中介、对撞都不在集内')

print()
print('注意 device_tier：它是纯结果预测变量。')
print(f'  在合法集里出现的次数: {sum(1 for v in valid if "device_tier" in v)}/{len(valid)}')
print('  它加不加都合法 —— 加上它不改变识别，但会**降低方差**。')
print('  这是八类变量里唯一「白拿」的一类（见正文的好控制/坏控制表）。')
print()
print('工程含义：')
print('  · 图是代码的一部分，改图需要 review，和改模型一样。')
print('  · 「多加一个特征」不再是一行改动 —— 必须先说明它在图里的位置。')
print('  · 图错了这套机制也会给出错的答案；它保证的是**假设可见**，不是假设正确。')